# lltInstructionData head cells on the Kaggle image

In [ ]:
# Runtime check. Run this cell first. It imports nothing heavy, so the install cell can still change versions.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU; Kaggle gives two, and the trainer refuses a DataParallel model
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less fragmentation when models come and go
import platform, shutil, subprocess, sys
from importlib.metadata import version, PackageNotFoundError
ok = True
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    ok = False
    print("Python 3.11 or later is needed, and 3.12 is recommended.")
for pkg in ["torch", "numpy", "transformers", "trl"]:
    try:
        print(pkg, version(pkg))
    except PackageNotFoundError:
        print(pkg, "not installed yet")
free = shutil.disk_usage(".").free / 1e9
print("Free disk", round(free, 1), "GB")
if free < 12:
    ok = False
    print("The data shards and five checkpoints need about 8 GB of disk. Free some space.")
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
except FileNotFoundError:                  # a CPU runtime has no nvidia-smi at all
    gpu = subprocess.CompletedProcess([], 1, "", "")
if gpu.returncode != 0 or not gpu.stdout.strip():
    ok = False
    print("No GPU. In Colab use Runtime > Change runtime type > T4 GPU. On Kaggle pick the T4 accelerator, not the P100.")
else:
    print("GPU", gpu.stdout.strip().splitlines()[0])
    if "P100" in gpu.stdout:
        ok = False
        print("Current PyTorch has no kernels for the P100. Switch the accelerator to T4.")
print("Ready." if ok else "Fix the items above, then run this cell again.")

In [ ]:
# Pinned versions of the recorded runs. torch and numpy stay as the runtime ships them: the constraints file stops pip from changing them.
from importlib.metadata import version
open("constraints.txt", "w").write("numpy==" + version("numpy") + chr(10) + "torch==" + version("torch") + chr(10))
%pip install -q -c constraints.txt "transformers==5.22.1" "tokenizers==0.23.3" "huggingface_hub==2.4.0" "datasets==5.1.0" "trl==1.17.1" "accelerate==1.15.0" "sentence-transformers==6.2.0" "lm-eval[ifeval]==0.4.13" "langdetect==1.0.9"

In [ ]:

import json, time, platform
from importlib.metadata import version
import torch, numpy, transformers, tokenizers, huggingface_hub, datasets, trl, accelerate, sentence_transformers, langdetect
from lm_eval.tasks.ifeval.utils import process_results
doc = {"key": 1, "prompt": "Write two sentences. Do not use any commas.", "instruction_id_list": ["punctuation:no_comma"], "kwargs": [{}]}
rec = {"python": platform.python_version(), "torch": torch.__version__, "numpy": numpy.__version__, "cuda": torch.cuda.is_available(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       **{p: version(p) for p in ["transformers", "tokenizers", "huggingface_hub", "datasets", "trl", "accelerate", "sentence-transformers", "lm_eval", "langdetect"]},
       "ifevalCheck": process_results(doc, ["Hello there. It is fine."])["prompt_level_strict_acc"]}
print(json.dumps(rec, indent=1))
json.dump(rec, open("insdata_head_check.json", "w"), indent=1)
